In [33]:
with open("the-verdict.txt","r") as f:
    raw_text=f.read()
print(len(raw_text))

20479


In [34]:
# import re 
# text="hello world! this is a test. let's see how it works."
# # re.split(r'(\s)',text)
# result=re.split(r'([,.!,",:;()\']|--|\s)',text)
# final=[res for res in result if res and not res.isspace()]
# final

In [35]:
# import re

# # Combined alternations outside the character class
# pattern = r'([.,!":;()\'?]|--|\s)'
# preprocessed_text = re.split(pattern, raw_text)

# final_preprocessed=[res for res in preprocessed_text if res and not res.isspace()]

In [36]:
# creating the token id 
# tokens=sorted(set(final_preprocessed))

In [37]:
# token_id={}
# for id , token in enumerate(tokens):
#     token_id[token]=id
    

In [ ]:
import re 
class Tokenizer:
    def __init__(self,data):
        self.data=data
    def tokenize(self):
        self.final_preprocessed=self.get_tokens(self.data)
        self.tokens=sorted(set(self.final_preprocessed))
        self.encode_token_id={token:id for id , token in enumerate(self.tokens)}
        self.decode_id_token={id:token for id , token in enumerate(self.tokens)}
        self.encode_token_id["<|endoftext|>"]=len(self.encode_token_id)+1
        self.decode_id_token[len(self.decode_id_token)+1]="<|endoftext|>"
        self.encode_token_id["<|unknown|>"]=len(self.encode_token_id)+2
        self.decode_id_token[len(self.decode_id_token)+2]="<|unknown|>"
    def get_tokens(self,data):
        pattern = r'([.,!":;()\'?]|--|\s)'
        preprocessed_text = re.split(pattern,data)
        final_preprocessed=[res for res in preprocessed_text if res and not res.isspace()]
        return final_preprocessed


    def encode(self,text):
        final_prep=self.get_tokens(text)
        tokenized_data=[]
        print(f"{len([unknown_token for unknown_token in final_prep if unknown_token not in self.encode_token_id])}")
        for tokens in final_prep:
            if tokens in self.encode_token_id.keys():
                tokenized_data.append(self.encode_token_id[tokens])
            else: 
                tokenized_data.append(self.encode_token_id["<|unknown|>"])

        return tokenized_data 
    def decode(self,tokenized_data):
        decoded_data=[]
        for token_id in tokenized_data:
            
            decoded_data.append(self.decode_id_token[token_id])
# Step 1: Join your data with spaces
        text = " ".join(decoded_data)

        # Step 2: Remove spaces BEFORE trailing punctuation (.,!:;!?) and double dashes
        text = re.sub(r'\s+([.,!:;!?]|--)', r'\1', text)

        # Step 3: Add a single space AFTER punctuation if it is missing
        text = re.sub(r'([.,!:;!?])(?=[^\s])', r'\1 ', text)

        # Step 4: Fix spaces inside parentheses and quotes
        text = re.sub(r'\(\s+', '(', text)  # Remove space after open parenthesis
        text = re.sub(r'\s+\)', ')', text)  # Remove space before close parenthesis

        # Step 5: Collapse any accidental double spaces created during the process
        decoded_text = re.sub(r'\s+', ' ', text).strip()

        return decoded_text   

In [39]:
tokenizer=Tokenizer(raw_text)
tokenizer.tokenize()

In [41]:
tokenizer.decode(tokenizer.encode("get out of my way, you are blocking the road!"))

2


'get out of my way, you are <|unknown|> the <|unknown|>!'

In [45]:
# subword tokenization technique.
#rule:1 Do not split  used words into smaller subwords.
# rule 2 : split the rare words into smallers subwords.
#-----------------------------------------------------------
# Byte pair encoding (BPE) algo : Most common pair is consecutive bytes of the data  is replaced with a byte that does not occur in data 
# mtlb for example yeh ek sequence hai "aabdaaabac"
# step1: find the most consecutive pair that is "aa" now replace it with z "aa"-->"z"
'''
new sequence will be "zabdzabac"
'''
# step 2: find another consecutive pair again in new sequence which is "ab"-->y
'''
new sequence will be "zydzyac"
'''
# again see the sequence and find the most common pair which is "zy"->x
'''
hence new sequence will be "xdxac"
'''
# here BPE is very chaotic to code so there is library named the tiktoken to perfom byte pair encoding 
# !pip install tiktoken

'\nhence new sequence will be "xdxac"\n'

In [46]:
import tiktoken
bpe=tiktoken.get_encoding('gpt2')

In [52]:
encoded_text=bpe.encode("Hello do you like tea <|end of text|>. i am veryhappy",allowed_special={"<|end of text|>"})
bpe.decode(encoded_text)

'Hello do you like tea <|end of text|>. i am veryhappy'